### RAG pipelines - Data ingestion to vactor DB pipeline

In [2]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, PyMuPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


c:\Users\vivek\Documents\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
### Read all the pdf files in the directory

def process_pdf_files(directory_path):
    all_documents = []
    pdf_directory = Path(directory_path)

    pdf_files = list(pdf_directory.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in the directory.")

    for pdf_file in pdf_files:
        print(f"\nProcessing file: {pdf_file}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            all_documents.extend(documents)

            
            print(f"\nSuccessfully loaded {len(documents)} documents from {pdf_file}")


        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    print(f"\nTotal documents loaded from PDF files: {len(all_documents)}")
    return all_documents


### Read all the text files in the directory
all_pdf_documents = process_pdf_files("../data")

Found 3 PDF files in the directory.

Processing file: ..\data\pdf\DUNGEON OF ETERNITY Report.pdf

Successfully loaded 37 documents from ..\data\pdf\DUNGEON OF ETERNITY Report.pdf

Processing file: ..\data\pdf\Lettre De Motivation.pdf

Successfully loaded 1 documents from ..\data\pdf\Lettre De Motivation.pdf

Processing file: ..\data\pdf\vivek_resume L.pdf

Successfully loaded 2 documents from ..\data\pdf\vivek_resume L.pdf

Total documents loaded from PDF files: 40


In [4]:
all_pdf_documents # Print the first 2 documents to verify

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-11-08T23:37:58+01:00', 'author': 'Vivek Jeevanaraj', 'moddate': '2024-11-08T23:37:58+01:00', 'source': '..\\data\\pdf\\DUNGEON OF ETERNITY Report.pdf', 'total_pages': 37, 'page': 0, 'page_label': '1', 'source_file': 'DUNGEON OF ETERNITY Report.pdf', 'file_type': 'pdf'}, page_content='1 \n \n \n \n \n \nDUNGEON OF ETERNITY  \n \n \nGroup-22  \nIsrat Jahan, Mostafizur Rahman, Noor-E- Alam, Pavel \nKhromov, Talha Aamer, Vivek Jeevanaraj \n \n \nNovember 08, 2024'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-11-08T23:37:58+01:00', 'author': 'Vivek Jeevanaraj', 'moddate': '2024-11-08T23:37:58+01:00', 'source': '..\\data\\pdf\\DUNGEON OF ETERNITY Report.pdf', 'total_pages': 37, 'page': 1, 'page_label': '2', 'source_file': 'DUNGEON OF ETERNITY Report.pdf', 'file

In [5]:
###Text Splitting get into smaller chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"\n split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print(f"\nFirst chunk content:\n{split_docs[0].page_content[:200]}...")  # Print the first 200 characters of the first chunk
        print(f"\n Metadata of the first chunk:\n{split_docs[0].metadata}")

    return split_docs


In [6]:
chunked_documents = split_documents(all_pdf_documents)
chunked_documents # Print the first 2 chunked documents to verify


 split 40 documents into 76 chunks.

First chunk content:
1 
 
 
 
 
 
DUNGEON OF ETERNITY  
 
 
Group-22  
Israt Jahan, Mostafizur Rahman, Noor-E- Alam, Pavel 
Khromov, Talha Aamer, Vivek Jeevanaraj 
 
 
November 08, 2024...

 Metadata of the first chunk:
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-11-08T23:37:58+01:00', 'author': 'Vivek Jeevanaraj', 'moddate': '2024-11-08T23:37:58+01:00', 'source': '..\\data\\pdf\\DUNGEON OF ETERNITY Report.pdf', 'total_pages': 37, 'page': 0, 'page_label': '1', 'source_file': 'DUNGEON OF ETERNITY Report.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-11-08T23:37:58+01:00', 'author': 'Vivek Jeevanaraj', 'moddate': '2024-11-08T23:37:58+01:00', 'source': '..\\data\\pdf\\DUNGEON OF ETERNITY Report.pdf', 'total_pages': 37, 'page': 0, 'page_label': '1', 'source_file': 'DUNGEON OF ETERNITY Report.pdf', 'file_type': 'pdf'}, page_content='1 \n \n \n \n \n \nDUNGEON OF ETERNITY  \n \n \nGroup-22  \nIsrat Jahan, Mostafizur Rahman, Noor-E- Alam, Pavel \nKhromov, Talha Aamer, Vivek Jeevanaraj \n \n \nNovember 08, 2024'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2024-11-08T23:37:58+01:00', 'author': 'Vivek Jeevanaraj', 'moddate': '2024-11-08T23:37:58+01:00', 'source': '..\\data\\pdf\\DUNGEON OF ETERNITY Report.pdf', 'total_pages': 37, 'page': 1, 'page_label': '2', 'source_file': 'DUNGEON OF ETERNITY Report.pdf', 'file

### Embedding and vector StoreDB

In [7]:
 import numpy as np
 from sentence_transformers import SentenceTransformer
 import chromadb
 from chromadb.config import Settings
 import uuid
 from typing import List, Dict, Any,Tuple
 from sklearn.metrics.pairwise import cosine_similarity


In [8]:
class EmbeddingManager:
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Successfully loaded embedding model: {self.model.get_sentence_embedding_dimension()} dimensions")
        except Exception as e:
            print(f"Error loading embedding model '{self.model_name}': {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        try:
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings for {len(texts)} texts.")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise


## init the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4785.63it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Successfully loaded embedding model: 384 dimensions


In [9]:
class VectorStore:
    def __init__(self, collection_name: str ="pdf_chunks", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata= {"description": "Collection of PDF document chunks"})
            print(f"Initialized vector store with collection: {self.collection_name}")
            print(f"Current number of documents in the collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):

        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must be the same.")
        
        print(f"Adding {len(documents)} documents to the vector store...")

        ids =[]
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = str(uuid.uuid4())
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding)

        try:
            self.collection.add(
                ids=ids,
                documents=documents_text,
                embeddings=embeddings_list,
                metadatas=metadatas
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in the collection after addition: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise 





In [10]:

vector_store = VectorStore()
vector_store

Initialized vector store with collection: pdf_chunks
Current number of documents in the collection: 0


In [11]:
## convert text chunks to embeddings and add to vector store

texts = [doc.page_content for doc in chunked_documents] # Print the first 5 chunked documents to verify


## Generate embeddings for the text chunks

embeddings = embedding_manager.generate_embeddings(texts)


#store the chunks and their embeddings in the vector store
vector_store.add_documents(chunked_documents, embeddings)


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.26s/it]


Generated embeddings for 76 texts.
Adding 76 documents to the vector store...
Successfully added 76 documents to the vector store.
Total documents in the collection after addition: 76
